# U-Net: предсказание маски на TSR-кропе 128×128

Неаугментированный датасет (`dataset.yaml` → `data/` + `labels/kaggle_binary_masks`),
4 окна 128×128 на ролик, признаки — `thermo.features.tsr.tsr_coeffs`.

Показывает: вход (канал TSR), GT, предсказание, **Dice** и **1 − Dice**.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path("..").resolve()
UNET = ROOT / "models" / "U-Net"
for p in (ROOT, UNET):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dataset import ThermalTSRDataset, load_dataset_yaml
from main import SoftDiceLoss, UNetModel

SAMPLE_INDEX = 0  # 0..len(ds)-1; каждый ролик даёт 4 кропа
CKPT = UNET / "model_unet_tsr_best.tar"
THRESHOLD = 0.5

cfg = load_dataset_yaml(UNET / "dataset.yaml")
ds = ThermalTSRDataset(cfg, augment=False, train=False)
print(f"len={len(ds)} (= {len(ds.stems)} videos × {ds.n_crops} crops)")
print(f"in_channels={ds.in_channels} | crop={ds.crop_size} | augment={ds.augment}")

In [ ]:
def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def dice_numpy(pred: np.ndarray, gt: np.ndarray, eps: float = 1e-6) -> float:
    p = pred.astype(bool).ravel()
    g = gt.astype(bool).ravel()
    inter = np.logical_and(p, g).sum()
    return float((2 * inter + eps) / (p.sum() + g.sum() + eps))


device = get_device()
model = UNetModel(in_channels=ds.in_channels, num_classes=1)
if CKPT.exists():
    state = torch.load(CKPT, map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    print("loaded", CKPT.name)
else:
    print("WARNING: checkpoint not found — using random weights:", CKPT)

model.eval().to(device)
bce = torch.nn.BCEWithLogitsLoss()
dice_loss = SoftDiceLoss()

In [ ]:
img, mask = ds[SAMPLE_INDEX]
video_i, crop_i = ds._index[SAMPLE_INDEX]
stem = ds.stems[video_i]
box = ds._boxes[crop_i]
print(f"sample={SAMPLE_INDEX} | {stem} crop#{crop_i} box=({box.y0}:{box.y1}, {box.x0}:{box.x1})")
print(f"img {tuple(img.shape)} | mask {tuple(mask.shape)}")

with torch.no_grad():
    logits = model(img.unsqueeze(0).to(device))
    loss_val = (
        bce(logits, mask.unsqueeze(0).to(device))
        + dice_loss(logits, mask.unsqueeze(0).to(device))
    ).item()
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()

pred = (prob > THRESHOLD).astype(np.float32)
gt = mask.squeeze().numpy()
dice = dice_numpy(pred, gt)
err = 1.0 - dice

print(f"Dice = {dice:.4f}")
print(f"error (1 − Dice) = {err:.4f}")
print(f"train-style loss (BCE+Dice) = {loss_val:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))

# row 0: a few TSR channels
for i, ax in enumerate(axes[0]):
    ch = min(i, img.shape[0] - 1)
    ax.imshow(img[ch].numpy(), cmap="inferno")
    ax.set_title(f"TSR ch{ch}")
    ax.axis("off")

axes[1, 0].imshow(gt, cmap="gray", vmin=0, vmax=1)
axes[1, 0].set_title("GT mask")
axes[1, 0].axis("off")

axes[1, 1].imshow(prob, cmap="magma", vmin=0, vmax=1)
axes[1, 1].set_title("Pred probability")
axes[1, 1].axis("off")

axes[1, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
axes[1, 2].contour(gt, levels=[0.5], colors="cyan", linewidths=0.8)
axes[1, 2].set_title(f"Pred binary\nDice={dice:.3f}  err={err:.3f}")
axes[1, 2].axis("off")

fig.suptitle(f"{stem} · crop {crop_i} · unaugmented", y=1.01)
plt.tight_layout()
plt.show()